# SAA count-table refactor: hand-checkable validation

This notebook is an executable test and readable demonstration of the implemented representation refactor. It uses one zone, three categorical attributes, six candidate household types, and 12 target households. It does **not** change the SAA algorithm or ILP.

The checks cover compression/expansion, the count-native SAA pass, the unchanged ILP contingency tables, conditional reconstruction, the legacy row backend, fixed seeds, and count-based removal. Every important claim below has an assertion.

In [1]:
import inspect
import random
from math import sqrt

import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display

from PopSynthesis.Methods.IPSF.const import count_field, zone_field
from PopSynthesis.Methods.IPSF.SAA.SAA import SAA
from PopSynthesis.Methods.IPSF.SAA.operations.count_tables import (
    aggregate_counts, counts_to_marginals, expand_counts, population_size,
    sample_without_replacement_counts,
)
from PopSynthesis.Methods.IPSF.SAA.operations.general import calculate_states_diff_count
from PopSynthesis.Methods.IPSF.SAA.operations.ILP_zone_ad import (
    convert_to_ILP_inputs, sample_partial_to_complete_counts,
)
from PopSynthesis.Methods.IPSF.SAA.operations.shared_vars import zero_cell_scope
from PopSynthesis.Methods.IPSF.SAA.operations.wrapper_saa_run import saa_run
from PopSynthesis.Methods.IPSF.utils.condensed import condense_df
from PopSynthesis.Methods.IPSF.utils.ILP_matrix_ad import (
    convert_back_to_syn_count, update_count_tables,
)

print('Refactored entry point:', inspect.signature(saa_run))
print('Count pass:', inspect.signature(SAA.run_count))
print('Expand adapter:', inspect.signature(expand_counts))
print('Compress adapter:', inspect.signature(condense_df))
print('ILP adapter:', inspect.signature(convert_to_ILP_inputs))
assert inspect.signature(saa_run).parameters['stochastic_mode'].default == 'count_native'
print('Legacy backend remains callable through stochastic_mode=legacy_exact.')

Refactored entry point: (targeted_marg: pandas.core.frame.DataFrame, count_pool: polars.dataframe.frame.DataFrame, considered_atts: Optional[List[str]] = None, ordered_to_adjust_atts: Optional[List[str]] = None, last_adjustment_order: Optional[List[str]] = None, max_run_time: int = 30, extra_rm_frac: float = 0, output_each_step: bool = False, add_name_for_step_output: str = '', include_zero_cell_values: bool = False, randomly_add_last: Optional[List[str]] = None, meta_output_dir: pathlib._local.Path = PosixPath('/Users/bobla/Documents/repos/PopSyn_Monash/PopSynthesis/Methods/IPSF/output/meta'), stochastic_mode: Literal['count_native', 'legacy_exact'] = 'count_native') -> Tuple[polars.dataframe.frame.DataFrame, List[int]]
Count pass: (self, include_zero_cell_values: bool = False) -> Tuple[polars.dataframe.frame.DataFrame, List[Tuple[int, str, polars.dataframe.frame.DataFrame, dict]]]
Expand adapter: (population: 'pl.DataFrame') -> 'pl.DataFrame'
Compress adapter: (df: Union[polars.dataf

## Toy candidate pool

Pool frequency is a sampling weight, not a capacity. The `Type` labels are for this notebook only; production SAA identifies a type by the composite attribute key.

In [2]:
ATTRIBUTES = ['hhsize', 'vehicles', 'income']
TYPE_KEYS = ['hhsize', 'vehicles', 'income']
pool_display = pd.DataFrame([
    ('A', '1',  '0',  'Low',  3),
    ('B', '1',  '1',  'High', 1),
    ('C', '2',  '1',  'Low',  2),
    ('D', '2',  '2+', 'High', 2),
    ('E', '3+', '1',  'Low',  1),
    ('F', '3+', '2+', 'High', 1),
], columns=['Type', *ATTRIBUTES, 'Frequency'])
display(pool_display)

pool_count = pl.from_pandas(
    pool_display.drop(columns='Type').rename(columns={'Frequency': count_field})
).with_columns(pl.col(count_field).cast(pl.Int64))
pool_before = pool_count.clone()

assert pool_count.columns == ATTRIBUTES + [count_field]
assert pool_count.schema[count_field].is_integer()
assert (pool_count[count_field] >= 0).all()
assert population_size(pool_count) == 10
assert pool_count.select(TYPE_KEYS).is_duplicated().sum() == 0

expanded_pool = expand_counts(pool_count)
recompressed_pool = condense_df(expanded_pool, maintain_order=True).sort(TYPE_KEYS)
assert len(expanded_pool) == 10
assert recompressed_pool.equals(pool_count.sort(TYPE_KEYS))
assert expanded_pool.columns == ATTRIBUTES
print('Pool round trip passed: 6 unique rows <-> 10 expanded rows.')

,Type,hhsize,vehicles,income,Frequency
0,A,1,0,Low,3
1,B,1,1,High,1
2,C,2,1,Low,2
3,D,2,2+,High,2
4,E,3+,1,Low,1
5,F,3+,2+,High,1


Pool round trip passed: 6 unique rows <-> 10 expanded rows.


## Scenario 1 — ordinary supported case

All states and required combinations are supported. Every marginal sums to 12. A hand solution is `A=3, B=1, C=3, D=2, E=1, F=2`.

Manual checks:
- household size: `(3+1, 3+2, 1+2) = (4, 5, 3)`;
- vehicles: `(3, 1+3+1, 2+2) = (3, 5, 4)`;
- income: `(3+3+1, 1+2+2) = (7, 5)`.

In [3]:
marginal_columns = pd.MultiIndex.from_tuples([
    ('hhsize', '1'), ('hhsize', '2'), ('hhsize', '3+'),
    ('vehicles', '0'), ('vehicles', '1'), ('vehicles', '2+'),
    ('income', 'Low'), ('income', 'High'),
])
target = pd.DataFrame(
    [[4, 5, 3, 3, 5, 4, 7, 5]],
    index=pd.Index([101], name=zone_field),
    columns=marginal_columns,
)
display(target)
assert set(target.T.groupby(level=0).sum().iloc[:, 0]) == {12}

golden = pl.from_pandas(
    pool_display.assign(count=[3, 1, 3, 2, 1, 2], zone_id='101')
    .drop(columns=['Frequency'])
    .rename(columns={'Type': 'type_id'})
).with_columns(pl.col(count_field).cast(pl.Int64))
golden_population = golden.drop('type_id')
golden_marginals = counts_to_marginals(golden_population, ATTRIBUTES)
target_for_compare = target.copy(); target_for_compare.index = target_for_compare.index.astype(str)
golden_marginals = golden_marginals.reindex(
    index=target_for_compare.index, columns=target_for_compare.columns, fill_value=0
)
pd.testing.assert_frame_equal(golden_marginals.astype(int), target_for_compare.astype(int), check_names=False)
display(golden.select(['type_id', *ATTRIBUTES, count_field]))

hhsize       vehicles       income     
             1  2 3+        0  1 2+    Low High
zone_id                                        
101          4  5  3        3  5  4      7    5

type_id,hhsize,vehicles,income,count
str,str,str,str,i64
"""A""","""1""","""0""","""Low""",3
"""B""","""1""","""1""","""High""",1
"""C""","""2""","""1""","""Low""",3
"""D""","""2""","""2+""","""High""",2
"""E""","""3+""","""1""","""Low""",1
"""F""","""3+""","""2+""","""High""",2


### Count-native pass and intermediate populations

The first table is the initialized compressed population. The next two are complete compressed populations after vehicle and income reconstruction.

In [4]:
random.seed(0); np.random.seed(0)
with zero_cell_scope():
    count_saa = SAA(target.copy(), ATTRIBUTES, ATTRIBUTES, pool_count)
    final_internal_count, adjustment_steps = count_saa.run_count(False)

assert len(adjustment_steps) == 3
for adjustment_index, attribute, population, zone_errors in adjustment_steps:
    print(f'After adjustment {adjustment_index}: {attribute}; zone errors={zone_errors}')
    display(population.sort([zone_field, *TYPE_KEYS]))
    assert population_size(population) == 12
    assert population.schema[count_field].is_integer()
    assert (population[count_field] >= 0).all()
    assert population.select([zone_field, *TYPE_KEYS]).is_duplicated().sum() == 0

final_internal_count = final_internal_count.with_columns(pl.col(zone_field).cast(pl.String))
assert final_internal_count.sort([zone_field, *TYPE_KEYS]).equals(
    golden_population.select(final_internal_count.columns).sort([zone_field, *TYPE_KEYS])
)
print('The count-native pass reached the hand-calculated golden joint counts exactly.')

After adjustment 0: hhsize; zone errors={}


hhsize,vehicles,income,zone_id,count
str,str,str,i32,i64
"""1""","""0""","""Low""",101,3
"""1""","""1""","""High""",101,1
"""2""","""1""","""Low""",101,3
"""2""","""2+""","""High""",101,2
"""3+""","""1""","""Low""",101,2
"""3+""","""2+""","""High""",101,1


After adjustment 1: vehicles; zone errors={'101': 0.0}


hhsize,vehicles,income,zone_id,count
str,str,str,i32,i64
"""1""","""0""","""Low""",101,3
"""1""","""1""","""High""",101,1
"""2""","""1""","""Low""",101,3
"""2""","""2+""","""High""",101,2
"""3+""","""1""","""Low""",101,1
"""3+""","""2+""","""High""",101,2


After adjustment 2: income; zone errors={'101': 0.0}


hhsize,vehicles,income,zone_id,count
str,str,str,i32,i64
"""1""","""0""","""Low""",101,3
"""1""","""1""","""High""",101,1
"""2""","""1""","""Low""",101,3
"""2""","""2+""","""High""",101,2
"""3+""","""1""","""Low""",101,1
"""3+""","""2+""","""High""",101,2


The count-native pass reached the hand-calculated golden joint counts exactly.


### Replay and expose the unchanged ILP tables

For each later attribute, the cell below reconstructs the exact pre-ILP contingency table, applies the existing `update_count_tables` function, and compares the unpivoted result with the known-attribute contingency after conditional reconstruction.

In [5]:
def stable_partial(df, keys):
    return aggregate_counts(df.select([*keys, count_field]), keys).sort(keys)

def replay_ilp(previous_population, reconstructed_population, active_attribute, adjusted_attributes):
    sub_census = pl.from_pandas(count_saa.segmented_marg[active_attribute].reset_index())
    diff_table = calculate_states_diff_count(active_attribute, previous_population, sub_census)
    diff = diff_table.row(0, named=True); diff.pop(zone_field)
    converted = convert_to_ILP_inputs(
        previous_population.drop(zone_field), active_attribute, adjusted_attributes,
        pool_count, maintain_order=True,
    ).with_row_index('row_id')
    previous_keys = converted.select([*adjusted_attributes, 'row_id'])
    state_table = converted.select([c for c in converted.columns if c not in adjusted_attributes])
    adjusted_states, error = update_count_tables(
        state_table, diff, 'row_id', deviation_type='absolute'
    )
    adjusted_wide = previous_keys.join(adjusted_states, on='row_id').drop('row_id')
    adjusted_partial = convert_back_to_syn_count(
        adjusted_wide, active_attribute, adjusted_attributes
    ).filter(pl.col(count_field) > 0)

    state_columns = [c for c in state_table.columns if c != 'row_id']
    before_rows = state_table.fill_null(0).select(pl.sum_horizontal(state_columns))
    after_rows = adjusted_states.fill_null(0).select(pl.sum_horizontal(state_columns))
    assert before_rows.equals(after_rows)
    remaining = {}
    for state in state_columns:
        before = int(state_table[state].fill_null(0).sum())
        after = int(adjusted_states[state].fill_null(0).sum())
        remaining[state] = int(diff[state] - (after - before))
    assert error == sqrt(sum(value ** 2 for value in remaining.values()))

    known_keys = [*adjusted_attributes, active_attribute]
    reconstructed_partial = reconstructed_population.group_by(known_keys, maintain_order=True).agg(
        pl.col(count_field).sum()
    )
    assert stable_partial(adjusted_partial, known_keys).equals(
        stable_partial(reconstructed_partial, known_keys)
    )
    assert population_size(adjusted_partial) == population_size(previous_population) == 12

    print(f'Active attribute: {active_attribute}; target difference={diff}; remaining={remaining}; slack norm={error}')
    print('Before ILP (rows are prior-attribute combinations):')
    display(converted.drop('row_id'))
    print('After ILP:')
    display(adjusted_wide)
    print('After conditional reconstruction, grouped by known attributes:')
    display(reconstructed_partial.sort(known_keys))
    return {'before': converted, 'after': adjusted_wide, 'remaining': remaining, 'error': error}

vehicle_ilp = replay_ilp(adjustment_steps[0][2], adjustment_steps[1][2], 'vehicles', ['hhsize'])
income_ilp = replay_ilp(adjustment_steps[1][2], adjustment_steps[2][2], 'income', ['hhsize', 'vehicles'])
assert vehicle_ilp['error'] == 0 and income_ilp['error'] == 0
assert all(value == 0 for value in vehicle_ilp['remaining'].values())
assert all(value == 0 for value in income_ilp['remaining'].values())

Active attribute: vehicles; target difference={'0': 0, '1': -1, '2+': 1}; remaining={'0': 0, '1': 0, '2+': 0}; slack norm=0.0
Before ILP (rows are prior-attribute combinations):


hhsize,0,1,2+
str,f64,i32,f64
"""1""",3.0,1,null
"""2""",null,3,2.0
"""3+""",null,2,1.0


After ILP:


hhsize,0,1,2+
str,f64,f64,f64
"""1""",3.0,1.0,null
"""2""",null,3.0,2.0
"""3+""",null,1.0,2.0


After conditional reconstruction, grouped by known attributes:


hhsize,vehicles,count
str,str,i64
"""1""","""0""",3
"""1""","""1""",1
"""2""","""1""",3
"""2""","""2+""",2
"""3+""","""1""",1
"""3+""","""2+""",2


... (2 outputs left out)

After ILP:


hhsize,vehicles,Low,High
str,str,f64,f64
"""1""","""0""",3.0,null
"""1""","""1""",null,1.0
"""2""","""1""",3.0,null
"""3+""","""1""",1.0,null
"""2""","""2+""",null,2.0
"""3+""","""2+""",null,2.0


After conditional reconstruction, grouped by known attributes:


hhsize,vehicles,income,count
str,str,str,i64
"""1""","""0""","""Low""",3
"""1""","""1""","""High""",1
"""2""","""1""","""Low""",3
"""2""","""2+""","""High""",2
"""3+""","""1""","""Low""",1
"""3+""","""2+""","""High""",2


### End-to-end count-native versus legacy expanded backend

Seed 0 is a deliberately retained exact comparison: both backends reach the same hand-calculated compressed joint table. Row order is not compared. Seed 1 demonstrates that stochastic joint counts can change while all required marginals remain exact.

In [6]:
def compress_output(expanded):
    return condense_df(expanded, maintain_order=True).sort([zone_field, *TYPE_KEYS])

def run_backend(mode, seed):
    random.seed(seed); np.random.seed(seed)
    expanded, targets = saa_run(
        target.copy(), pool_count, considered_atts=ATTRIBUTES,
        ordered_to_adjust_atts=ATTRIBUTES, last_adjustment_order=ATTRIBUTES,
        max_run_time=1, stochastic_mode=mode,
    )
    return expanded, compress_output(expanded), targets

native_rows_1, native_count_1, native_targets_1 = run_backend('count_native', 0)
native_rows_2, native_count_2, native_targets_2 = run_backend('count_native', 0)
legacy_rows, legacy_count, legacy_targets = run_backend('legacy_exact', 0)
different_seed_rows, different_seed_count, _ = run_backend('count_native', 1)

golden_sorted = golden_population.select(native_count_1.columns).sort([zone_field, *TYPE_KEYS])
assert native_count_1.equals(native_count_2)
assert native_targets_1 == native_targets_2 == legacy_targets == [12]
assert native_count_1.equals(legacy_count)
assert native_count_1.equals(golden_sorted)
assert not different_seed_count.equals(native_count_1)
assert len(native_rows_1) == len(legacy_rows) == len(different_seed_rows) == 12
assert pool_count.equals(pool_before)  # neither backend mutated the shared candidate pool

for label, compressed in [('native seed 0', native_count_1), ('legacy seed 0', legacy_count), ('native seed 1', different_seed_count)]:
    observed = counts_to_marginals(compressed, ATTRIBUTES).reindex(
        index=target_for_compare.index, columns=target_for_compare.columns, fill_value=0
    )
    pd.testing.assert_frame_equal(observed.astype(int), target_for_compare.astype(int), check_names=False)
    print(label)
    display(compressed)

assert set(native_rows_1.columns) == set([*ATTRIBUTES, zone_field])
assert set(native_count_1.columns) == set([*ATTRIBUTES, zone_field, count_field])
print('Exact seed-0 compressed equivalence and same-seed reproducibility passed.')

ADJUSTING FOR hhsize
ADJUSTING FOR vehicles
DOING zone 101
ADJUSTING FOR income
DOING zone 101
native seed 0


hhsize,vehicles,income,zone_id,count
str,str,str,str,u32
"""1""","""0""","""Low""","""101""",3
"""1""","""1""","""High""","""101""",1
"""2""","""1""","""Low""","""101""",3
"""2""","""2+""","""High""","""101""",2
"""3+""","""1""","""Low""","""101""",1
"""3+""","""2+""","""High""","""101""",2


legacy seed 0


hhsize,vehicles,income,zone_id,count
str,str,str,str,u32
"""1""","""0""","""Low""","""101""",3
"""1""","""1""","""High""","""101""",1
"""2""","""1""","""Low""","""101""",3
"""2""","""2+""","""High""","""101""",2
"""3+""","""1""","""Low""","""101""",1
"""3+""","""2+""","""High""","""101""",2


native seed 1


hhsize,vehicles,income,zone_id,count
str,str,str,str,u32
"""1""","""0""","""Low""","""101""",3
"""1""","""1""","""High""","""101""",1
"""2""","""1""","""Low""","""101""",4
"""2""","""2+""","""High""","""101""",1
"""3+""","""2+""","""High""","""101""",3


Exact seed-0 compressed equivalence and same-seed reproducibility passed.


## Scenario 2 — one eligible type, requested count larger than pool frequency

For `(hhsize=3+, vehicles=1)`, only type E is eligible and its pool frequency is 1. Requesting 5 records must still produce five E records because conditional pool sampling is with replacement. This exercises an edge case supported by the current implementation without inventing new zero-cell behaviour.

In [7]:
partial = pl.DataFrame({'hhsize': ['3+'], 'vehicles': ['1'], count_field: [5]})
np.random.seed(44)
completed = sample_partial_to_complete_counts(partial, pool_count)
display(completed)
assert population_size(completed) == 5
assert completed.height == 1
assert completed.row(0, named=True) == {
    'hhsize': '3+', 'vehicles': '1', 'income': 'Low', count_field: 5,
}
assert pool_count.equals(pool_before)
print('One eligible record with pool frequency 1 correctly supplied count 5 via replacement.')

hhsize,vehicles,income,count
str,str,str,i64
"""3+""","""1""","""Low""",5


One eligible record with pool frequency 1 correctly supplied count 5 via replacement.


## Focused count-based removal test

Starting from the 12-household golden table, retain 7 (remove 5). Seed 1 completely removes type A and partially reduces type C from 3 to 1.

In [8]:
removal_source = golden.select(['type_id', *ATTRIBUTES, count_field])
np.random.seed(1)
retained_1 = sample_without_replacement_counts(removal_source, 7).sort('type_id')
np.random.seed(1)
retained_2 = sample_without_replacement_counts(removal_source, 7).sort('type_id')
display(retained_1)
assert retained_1.equals(retained_2)
assert population_size(removal_source) - population_size(retained_1) == 5
assert population_size(retained_1) == 7
assert retained_1.schema[count_field].is_integer()
assert (retained_1[count_field] >= 0).all()
assert 'A' not in retained_1['type_id'].to_list()  # complete removal
assert retained_1.filter(pl.col('type_id') == 'C')[count_field].item() == 1  # partial removal
assert len(expand_counts(retained_1)) == 7
assert set(expand_counts(retained_1).columns) == set(['type_id', *ATTRIBUTES])
print('Count-based removal is exact, non-negative, and reproducible.')

type_id,hhsize,vehicles,income,count
str,str,str,str,i64
"""B""","""1""","""1""","""High""",1
"""C""","""2""","""1""","""Low""",1
"""D""","""2""","""2+""","""High""",2
"""E""","""3+""","""1""","""Low""",1
"""F""","""3+""","""2+""","""High""",2


Count-based removal is exact, non-negative, and reproducible.


## Notebook-contained automated test suite

This `unittest` suite turns the demonstrated results into a conventional automated pass/fail summary without creating a second script or external fixture.

In [9]:
import unittest

class SAARefactorNotebookTests(unittest.TestCase):
    def test_count_schema_and_round_trip(self):
        self.assertTrue(pool_count.schema[count_field].is_integer())
        self.assertTrue((pool_count[count_field] >= 0).all())
        self.assertEqual(len(expanded_pool), population_size(pool_count))
        self.assertTrue(recompressed_pool.equals(pool_count.sort(TYPE_KEYS)))

    def test_hand_calculated_golden_and_ilp(self):
        self.assertTrue(native_count_1.equals(golden_sorted))
        self.assertEqual(vehicle_ilp['error'], 0)
        self.assertEqual(income_ilp['error'], 0)

    def test_legacy_and_reproducibility(self):
        self.assertTrue(native_count_1.equals(legacy_count))
        self.assertTrue(native_count_1.equals(native_count_2))
        self.assertFalse(native_count_1.equals(different_seed_count))

    def test_edge_reconstruction_and_removal(self):
        self.assertEqual(population_size(completed), 5)
        self.assertEqual(population_size(retained_1), 7)
        self.assertTrue(retained_1.equals(retained_2))

    def test_shared_pool_and_public_columns(self):
        self.assertTrue(pool_count.equals(pool_before))
        self.assertEqual(set(native_rows_1.columns), set([*ATTRIBUTES, zone_field]))

suite = unittest.defaultTestLoader.loadTestsFromTestCase(SAARefactorNotebookTests)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful()


test_count_schema_and_round_trip (__main__.SAARefactorNotebookTests.test_count_schema_and_round_trip) ... ok
test_edge_reconstruction_and_removal (__main__.SAARefactorNotebookTests.test_edge_reconstruction_and_removal) ... ok
test_hand_calculated_golden_and_ilp (__main__.SAARefactorNotebookTests.test_hand_calculated_golden_and_ilp) ... ok
test_legacy_and_reproducibility (__main__.SAARefactorNotebookTests.test_legacy_and_reproducibility) ... ok
test_shared_pool_and_public_columns (__main__.SAARefactorNotebookTests.test_shared_pool_and_public_columns) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.005s

OK


## Representation sanity check and conclusions

This is intentionally not a performance benchmark. It only shows that repeated individual rows can be represented by fewer unique rows plus integer counts. Production SAA expands at the public final-output boundary; the count-native pass itself performs no population expansion.

In [10]:
benefit = pd.DataFrame([
    {
        'object': 'candidate pool', 'expanded_rows': len(expanded_pool),
        'unique_count_rows': pool_count.height,
        'expanded_bytes': expanded_pool.estimated_size(),
        'count_table_bytes': pool_count.estimated_size(),
    },
    {
        'object': 'seed-0 final population', 'expanded_rows': len(native_rows_1),
        'unique_count_rows': native_count_1.height,
        'expanded_bytes': native_rows_1.estimated_size(),
        'count_table_bytes': native_count_1.estimated_size(),
    },
])
display(benefit)
assert (benefit['unique_count_rows'] < benefit['expanded_rows']).all()

summary = pd.DataFrame({
    'check': [
        'count/expanded round trip', 'population total', 'ILP row/column invariants',
        'conditional reconstruction', 'legacy exact compressed result',
        'same-seed reproducibility', 'different-seed invariants', 'count removal',
        'candidate pool unchanged',
    ],
    'result': ['PASS'] * 9,
})
display(summary)
print('Logical expansion/compression boundaries in the basic demonstration:')
print('- explicit pool round trip: 1 expansion + 1 compression')
print('- count-native SAA internals: 0 population expansions')
print('- public final output: 1 expansion')
print('ALL SAA COUNT-TABLE DEMONSTRATION ASSERTIONS PASSED')

,object,expanded_rows,unique_count_rows,expanded_bytes,count_table_bytes
0,candidate pool,10,6,59,85
1,seed-0 final population,12,6,108,79


,check,result
0,count/expanded round trip,PASS
1,population total,PASS
2,ILP row/column invariants,PASS
3,conditional reconstruction,PASS
4,legacy exact compressed result,PASS
5,same-seed reproducibility,PASS
6,different-seed invariants,PASS
7,count removal,PASS
8,candidate pool unchanged,PASS


Logical expansion/compression boundaries in the basic demonstration:
- explicit pool round trip: 1 expansion + 1 compression
- count-native SAA internals: 0 population expansions
- public final output: 1 expansion
ALL SAA COUNT-TABLE DEMONSTRATION ASSERTIONS PASSED
